# Transforms

In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

ds = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

In [3]:
target_transform = Lambda(lambda y: torch.zeros(
    10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))
target_transform(4)

tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])

# Build the Neural Network

In [5]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [10]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [17]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(logits)
pred_probab = nn.Softmax(dim=1)(logits)
print(pred_probab)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

tensor([[ 0.0520, -0.0585,  0.0139,  0.0104,  0.1191, -0.0150, -0.0614, -0.0159,
         -0.0497,  0.0532]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[0.1047, 0.0937, 0.1008, 0.1004, 0.1119, 0.0979, 0.0935, 0.0978, 0.0946,
         0.1048]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([4], device='cuda:0')


## Model Layers

In [18]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


### nn.Flatten

In [19]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


### nn.Linear

全连接层

$$output=X⋅W^T+b$$


- X : 输入张量，shape (batch_size, 784)
- W : 权重矩阵，shape (20, 784)
- b : 偏置项，shape (20,)
- 输出 shape 就是 (batch_size, 20)

In [20]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


### nn.ReLU

线性层后面跟一个激活函数：ReLU

就有了非线性的

In [21]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.6198,  0.0022,  0.1497, -0.0269,  0.0446, -0.4677, -0.0317, -0.3697,
         -0.0941,  0.1559,  0.0825, -0.3023, -0.1279, -0.1033,  0.0169, -0.1344,
          0.0198, -0.2843, -0.0234,  0.3441],
        [ 0.3813, -0.0031,  0.2042,  0.0044,  0.5143, -0.3612, -0.4462, -0.6563,
         -0.2300,  0.2184, -0.1153, -0.0232, -0.3222,  0.0236, -0.0897, -0.1710,
         -0.1725, -0.3513,  0.3598,  0.7864],
        [ 0.4414,  0.1081,  0.0892,  0.2675, -0.0075, -0.2284, -0.3953, -0.3478,
         -0.1866,  0.3794,  0.2067, -0.2716, -0.1457, -0.0225, -0.1690, -0.2028,
         -0.1577, -0.1257,  0.1018,  0.8275]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.6198, 0.0022, 0.1497, 0.0000, 0.0446, 0.0000, 0.0000, 0.0000, 0.0000,
         0.1559, 0.0825, 0.0000, 0.0000, 0.0000, 0.0169, 0.0000, 0.0198, 0.0000,
         0.0000, 0.3441],
        [0.3813, 0.0000, 0.2042, 0.0044, 0.5143, 0.0000, 0.0000, 0.0000, 0.0000,
         0.2184, 0.0000, 0.0000, 0.0000, 0.0236, 0.00

### nn.Sequential

In [23]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.1480, -0.0482,  0.1017,  0.2075,  0.1926, -0.2546,  0.0425,  0.0077,
          0.1509, -0.1208],
        [ 0.0616,  0.0803,  0.2274,  0.0458,  0.2871, -0.2684,  0.0212,  0.0941,
          0.0868, -0.1651],
        [ 0.0966, -0.0380,  0.0843,  0.0302,  0.2656, -0.1298,  0.0146, -0.0751,
          0.0292, -0.0604]], grad_fn=<AddmmBackward0>)


### nn.Softmax

In [24]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

## Model Parameters

In [25]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0139, -0.0104, -0.0341,  ..., -0.0270, -0.0022, -0.0341],
        [ 0.0194, -0.0073,  0.0166,  ...,  0.0300, -0.0262,  0.0143]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0230, -0.0312], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0275,  0.0334,  0.0278,  ...,  0.0379, -0.0375,  0.0184],
        [-0.0125,  0.0304, -0.0387,  ..., -0.0210,  0.0111, -0.0204]],
       device='cuda:0', grad_fn=<Sl